# Splity akcji

**Pytanie:** które zmiany liczby akcji między kwartałami to splity (lub odwrotne splity), a nie kupno albo sprzedaż?

**Wniosek:** do uzupełnienia po decyzji w ostatniej sekcji. Progi trafiają do `docs/decisions.md`.

Metoda: dla każdego papieru i kwartału liczymy medianę po filerach z `shares / prev_shares` (`shr_ratio`) i `value_corrected / prev_value` (`val_ratio`). Przy splicie prawie wszyscy posiadacze mają tę samą zmianę liczby akcji, a wartość pozycji zmienia się tylko o ruch kursu.

Dane: `positions_pit.parquet`. Porównujemy tylko kolejne kwartały tego samego filera.

In [1]:
import duckdb

con = duckdb.connect()
con.sql("SET preserve_insertion_order = false")
%load_ext sql
%sql con
%config SqlMagic.displaylimit = 30

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Całość naraz nie mieści się w pamięci (okno i mediana na 28 mln wierszy), więc liczymy w 8 częściach według `hash(CUSIP)`. Każdy papier jest w całości w jednej części, więc wynik jest taki sam.

In [2]:
RATIOS = """
WITH changes AS (
    SELECT
        PERIODOFREPORT, CUSIP, shares, value_corrected,
        lag(shares) OVER w AS prev_shares,
        lag(value_corrected) OVER w AS prev_value,
        lag(PERIODOFREPORT) OVER w AS prev_period
    FROM '../data/processed/positions_pit.parquet'
    WHERE hash(CUSIP) % 8 = {part}
    WINDOW w AS (PARTITION BY CIK, CUSIP ORDER BY PERIODOFREPORT)
)
SELECT
    PERIODOFREPORT, CUSIP,
    median(shares * 1.0 / prev_shares) AS shr_ratio,
    median(value_corrected * 1.0 / prev_value) AS val_ratio,
    count(*) AS filers_ct
FROM changes
WHERE prev_shares > 0 AND prev_value > 0
  AND PERIODOFREPORT = last_day(prev_period + INTERVAL 3 MONTH)
GROUP BY PERIODOFREPORT, CUSIP
"""

con.sql("CREATE OR REPLACE TABLE ratios AS " + RATIOS.format(part=0))
for part in range(1, 8):
    con.sql("INSERT INTO ratios " + RATIOS.format(part=part))
con.sql("SELECT count(*) AS pary FROM ratios")

paryBIGINT
238347


Kontrola: NVIDIA miała split 10:1 w czerwcu 2024, SPY nie miał żadnego.

In [3]:
%%sql
SELECT CUSIP, PERIODOFREPORT, round(shr_ratio, 3) AS shr, round(val_ratio, 3) AS val, filers_ct
FROM ratios
WHERE CUSIP IN ('67066G104', '78462F103') AND PERIODOFREPORT BETWEEN '2024-03-31' AND '2024-09-30'
ORDER BY CUSIP, PERIODOFREPORT

Running query in 'DuckDBPyConnection'

CUSIP,PERIODOFREPORT,shr,val,filers_ct
67066G104,2024-03-31,0.996,1.808,3560
67066G104,2024-06-30,10.005,1.369,3924
67066G104,2024-09-30,1.003,0.988,4144
78462F103,2024-03-31,1.0,1.1,3208
78462F103,2024-06-30,1.0,1.04,3337
78462F103,2024-09-30,1.0,1.054,3406


## 1. Ilu filerów potrzeba, żeby wierzyć medianie?

Liczba par przy różnych progach. Mniejszy próg daje więcej papierów, ale mediana z kilku filerów jest przypadkowa.

In [4]:
%%sql
SELECT prog, count(*) FILTER (WHERE filers_ct >= prog) AS pary,
       count(*) FILTER (WHERE filers_ct >= prog AND (shr_ratio >= 1.4 OR shr_ratio <= 1 / 1.4)) AS daleko_od_1
FROM ratios, (VALUES (1), (5), (10), (20), (50)) t(prog)
GROUP BY prog ORDER BY prog

Running query in 'DuckDBPyConnection'

prog,pary,daleko_od_1
1,238347,19894
5,132350,2016
10,114343,953
20,95721,530
50,69815,264


## 2. Jak daleko od 1 leżą mediany?

Prawie wszystkie pary są przy 1, bo normalny handel nie zmienia mediany filerów.

In [5]:
%%sql
SELECT CASE WHEN shr_ratio < 1 / 1.5 THEN 'a: < 0,67'
            WHEN shr_ratio < 0.9 THEN 'b: 0,67-0,9'
            WHEN shr_ratio <= 1.1 THEN 'c: 0,9-1,1'
            WHEN shr_ratio < 1.5 THEN 'd: 1,1-1,5'
            ELSE 'e: >= 1,5' END AS przedzial,
       count(*) AS pary
FROM ratios WHERE filers_ct >= 20
GROUP BY 1 ORDER BY 1

Running query in 'DuckDBPyConnection'

przedzial,pary
"a: < 0,67",32
"b: 0,67-0,9",164
"c: 0,9-1,1",93448
"d: 1,1-1,5",1678
"e: >= 1,5",399


## 3. Czy kandydaci wyglądają jak splity?

Prawdziwe splity mają okrągły `shr_ratio` (2, 3, 10) i `val_ratio` blisko 1.

In [6]:
%%sql
SELECT PERIODOFREPORT, CUSIP, round(shr_ratio, 3) AS shr, round(val_ratio, 3) AS val, filers_ct
FROM ratios
WHERE filers_ct >= 20 AND shr_ratio >= 1.4
ORDER BY filers_ct DESC LIMIT 20

Running query in 'DuckDBPyConnection'

PERIODOFREPORT,CUSIP,shr,val,filers_ct
2024-06-30,67066G104,10.005,1.369,3924
2025-12-31,64110L106,10.0,0.782,2953
2024-09-30,11135F101,10.0,1.076,2923
2024-03-31,931142103,3.0,1.145,2712
2026-06-30,922908736,6.0,1.183,2549
2026-06-30,464287614,4.0,1.165,2274
2026-06-30,922908629,4.0,1.122,2228
2025-12-31,81369Y803,2.0,1.022,2056
2026-06-30,482480100,10.0,2.044,1906
2024-03-31,464287507,5.0,1.096,1904


## 4. Sztywne widełki na `val_ratio` czy porównanie z `shr_ratio`?

Przy kupowaniu `shr_ratio` i `val_ratio` rosną podobnie. Przy splicie `shr_ratio` rośnie dużo bardziej.

In [7]:
%%sql
SELECT PERIODOFREPORT, CUSIP, round(shr_ratio, 3) AS shr, round(val_ratio, 3) AS val,
       round(shr_ratio / val_ratio, 2) AS shr_do_val, filers_ct
FROM ratios
WHERE filers_ct >= 20 AND shr_ratio >= 1.4 AND val_ratio NOT BETWEEN 0.5 AND 2
ORDER BY filers_ct DESC LIMIT 20

Running query in 'DuckDBPyConnection'

PERIODOFREPORT,CUSIP,shr,val,shr_do_val,filers_ct
2026-06-30,482480100,10.0,2.044,4.89,1906
2026-03-31,78454L100,1.728,2.811,0.61,277
2025-09-30,09290C780,2.335,2.746,0.85,243
2024-03-31,65290E101,1.841,2.195,0.84,192
2026-06-30,314911108,1.605,2.46,0.65,161
2023-12-31,05990K106,2.225,2.414,0.92,153
2026-06-30,81761L102,1.983,2.474,0.8,137
2023-09-30,49177J102,4.799,3.648,1.32,133
2025-06-30,09290C806,5.442,6.11,0.89,113
2026-03-31,09253N104,2.091,2.009,1.04,98


## 5. Strefa graniczna: 1,1-1,5

Tu są splity 3:2 (dokładnie 1,5) i fuzje, w których posiadacze dostają akcje przejmującego w nieokrągłej proporcji. Fuzje to faktyczna zmiana pozycji, nie split.

In [8]:
%%sql
SELECT PERIODOFREPORT, CUSIP, round(shr_ratio, 3) AS shr, round(val_ratio, 3) AS val, filers_ct
FROM ratios
WHERE filers_ct >= 20 AND shr_ratio BETWEEN 1.1 AND 1.6
ORDER BY filers_ct DESC LIMIT 20

Running query in 'DuckDBPyConnection'

PERIODOFREPORT,CUSIP,shr,val,filers_ct
2025-06-30,14040H105,1.386,1.647,1268
2026-06-30,25179M103,1.303,1.074,1074
2025-12-31,681919106,1.176,1.159,813
2025-12-31,11271J107,1.5,1.004,760
2023-03-31,693718108,1.5,1.109,683
2026-03-31,941848103,1.459,1.143,682
2024-09-30,084423102,1.5,1.083,661
2024-12-31,46438F101,1.156,1.689,589
2025-06-30,G0250X107,1.159,1.098,569
2024-12-31,594972408,1.115,1.897,559


## 6. Spadki: odwrotne splity

Przy odwrotnym splicie `shr_ratio` spada (np. 0,1 dla 1:10), a `val_ratio` zostaje przy 1. Gdy spadają oba, to nie jest split.

In [9]:
%%sql
SELECT PERIODOFREPORT, CUSIP, round(shr_ratio, 3) AS shr, round(val_ratio, 3) AS val, filers_ct
FROM ratios
WHERE filers_ct >= 20 AND shr_ratio <= 0.9
ORDER BY shr_ratio LIMIT 30

Running query in 'DuckDBPyConnection'

PERIODOFREPORT,CUSIP,shr,val,filers_ct
2026-03-31,G0250X107,0.199,0.947,26
2026-03-31,G0250X149,0.213,1.007,38
2023-06-30,34417J104,0.219,0.13,24
2026-06-30,26614N102,0.333,0.987,48
2023-06-30,Y28895103,0.367,0.296,20
2025-06-30,G7606H108,0.393,0.382,26
2025-09-30,G2949D104,0.397,0.406,24
2023-06-30,G5000D103,0.402,0.412,20
2026-03-31,210322509,0.427,0.43,24
2023-03-31,G0112R108,0.439,0.448,21


## 7. Decyzja

Wpisz swoje progi (liczba filerów, `shr_ratio` w górę i w dół, warunek na wartość), policz kandydatów i przejrzyj kilka ręcznie. Potem przenieś warunki do `src/radar/sql/splits.sql` i zapisz uzasadnienie w `docs/decisions.md`.

In [10]:
%%sql
SELECT
    PERIODOFREPORT, CUSIP,
    round(shr_ratio, 3) AS shr, round(val_ratio, 3) AS val, filers_ct,
    greatest(shr_ratio, 1 / shr_ratio) AS unified_shr,
    CASE WHEN shr_ratio >= 1
        THEN shr_ratio / val_ratio
        ELSE val_ratio / shr_ratio
    END AS sv
FROM ratios
WHERE filers_ct >= 10
  AND unified_shr > 1.4
  AND sv > 1.3
  AND abs(unified_shr * 2 - round(unified_shr * 2)) / unified_shr < 0.02
ORDER BY filers_ct DESC

Running query in 'DuckDBPyConnection'

PERIODOFREPORT,CUSIP,shr,val,filers_ct,unified_shr,sv
2024-06-30,67066G104,10.005,1.369,3924,10.005181582938064,7.308699754782611
2025-12-31,64110L106,10.0,0.782,2953,10.0,12.78710675936841
2024-09-30,11135F101,10.0,1.076,2923,10.0,9.291051576453436
2024-03-31,931142103,3.0,1.145,2712,3.0,2.620067937545351
2026-06-30,922908736,6.0,1.183,2549,6.0,5.070698862317158
2026-06-30,464287614,4.0,1.165,2274,4.0,3.4340050607809327
2026-06-30,922908629,4.0,1.122,2228,4.0,3.5643527314759633
2025-12-31,81369Y803,2.0,1.022,2056,2.0,1.9577698243154311
2026-06-30,482480100,10.0,2.044,1906,10.0,4.893396685951589
2024-03-31,464287507,5.0,1.096,1904,5.0,4.562892097864917
